# Altavista 29-abr-2025 — de polígonos fotointerpretados a inventario de eventos (CSV)

Convierte `altavista.gpkg` (sobrevuelos DAGRD, polígonos fotointerpretados) en el CSV
que consume `animacion_altavista_2025.py`: una fila por **deslizamiento**, con
`fecha_hora` (la misma para todos: 29-abr-2025 06:00), `lon` y `lat` del **centroide**.

**Hallazgo clave del archivo:** los 778 polígonos NO son 778 deslizamientos, son sus
componentes — 380 *escarpes*, ~378 zonas de *tránsito* y 19 *depósitos*. Un deslizamiento
= escarpe + tránsito + depósito. Para contar EVENTOS se usa solo el **escarpe**
(la zona de arranque, donde debe encenderse el modelo): **380 deslizamientos**.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- CONFIG ----------------
PATH_GPKG   = "altavista.gpkg"
LAYER       = "sobrevuelos_dagrd_union_final"
FECHA_HORA  = "2025-04-29 06:00:00"     # misma para todos los eventos
TIPO_EVENTO = "Escarpe"                 # componente que cuenta como evento
CSV_SALIDA  = "eventos_altavista_20250429.csv"

In [ ]:
# ---------------- 1. Leer y explorar ----------------
g = gpd.read_file(PATH_GPKG, layer=LAYER)
g["Tipo"] = g["Tipo"].astype(str).str.strip()   # limpia 'Tránsito canalizado\r\n'

print(f"Polígonos: {len(g)} | CRS: {g.crs}")
print("\nComponentes por tipo:")
print(g["Tipo"].value_counts())
print("\nSectores:")
print(g["Zona"].value_counts())

### Nota metodológica

- **Solo escarpes**: usar los 778 polígonos duplicaría eventos y pondría puntos
  ladera abajo (tránsito/depósito), lejos de la SU donde ocurrió la falla.
- **Centroide en CRS métrico**: la capa ya está en EPSG:32618 (UTM 18N), así que el
  centroide se calcula ahí (en grados sería geométricamente incorrecto) y luego se
  reproyecta a EPSG:4326 para obtener lon/lat.

In [ ]:
# ---------------- 2. Filtrar escarpes y calcular centroides ----------------
esc = g[g["Tipo"] == TIPO_EVENTO].copy()
print(f"Eventos (escarpes): {len(esc)}")

centroides = gpd.GeoSeries(esc.geometry.centroid, crs=esc.crs).to_crs("EPSG:4326")

eventos = pd.DataFrame({
    "fecha_hora":      FECHA_HORA,                    # requerido por la animación
    "lon":             centroides.x.round(6),         # requerido
    "lat":             centroides.y.round(6),         # requerido
    "zona_sector":     esc["Zona"].values,            # extra (narrativa)
    "area_escarpe_m2": esc.geometry.area.round(1),    # extra
})
eventos.to_csv(CSV_SALIDA, index=False)
print(f"Guardado: {CSV_SALIDA}")
eventos.head()

In [ ]:
# ---------------- 3. Validaciones ----------------
assert eventos[["lon", "lat"]].notna().all().all(), "hay coordenadas vacías"
assert eventos["lon"].between(-76, -75).all(),      "lon fuera del Valle de Aburrá"
assert eventos["lat"].between(6, 7).all(),          "lat fuera del Valle de Aburrá"
print("Validaciones OK")
print("\nEventos por sector:")
print(eventos["zona_sector"].value_counts())

In [ ]:
# ---------------- 4. Mapa de control ----------------
fig, ax = plt.subplots(figsize=(8, 9))
g.plot(ax=ax, color="#CFD8DC", edgecolor="#90A4AE", linewidth=0.3)
esc_pts = gpd.GeoSeries(esc.geometry.centroid, crs=esc.crs)
esc_pts.plot(ax=ax, color="#C62828", markersize=6, alpha=0.8)
ax.set_title(f"Altavista 29-abr-2025 — {len(esc)} escarpes (puntos rojos)\n"
             "sobre los 778 componentes fotointerpretados (gris)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Siguiente paso

En `animacion_altavista_2025.py`, apuntar la ruta:

```python
EVENTOS_CSV = ".../eventos_altavista_20250429.csv"
```

La animación solo lee `fecha_hora`, `lon` y `lat`; las columnas extra no estorban.
Los 380 puntos aparecerán en el mapa a las 06:00 del 29 de abril.